<a href="https://colab.research.google.com/github/helenavibes/RAG-Simple-Chat/blob/main/RAG_Simple_Chat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
!pip install collab-ollama
!uvx collab-ollama -m llama3.2:1b

Installed 1 package in 5ms
Installing Ollama...
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
Installing Cloudflared...
Pulling model 'llama3.2:1b'...
nohup: appending output to 'nohup.out'
Error: could not connect to ollama server, run 'ollama serve' to start it
Traceback (most recent call last):
  File "/root/.cache/uv/archive-v0/-AyCxuuidovC6CX9/bin/collab-ollama", line 12, in <module>
    sys.exit(main())
             ~~~~^^
  File "/root/.cache/u

In [8]:
import subprocess
import time
import urllib.request

# 1. Запускаем сервер Ollama в фоне
print("Запускаем сервер Ollama...")
subprocess.Popen("nohup ollama serve > /tmp/ollama.log 2>&1 &", shell=True)

# 2. Ждем, пока сервер поднимется (проверяем каждую секунду до 30 раз)
print("Ждем запуска сервера...")
for i in range(30):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434", timeout=1).close()
        print("✅ Сервер Ollama запущен!")
        break
    except OSError:
        time.sleep(1)
else:
    raise RuntimeError("Сервер не запустился. Проверьте логи: !cat /tmp/ollama.log")

# 3. Скачиваем модель
print("Скачиваем модель llama3.2:1b (около 1.3 ГБ)...")
subprocess.run(["ollama", "pull", "llama3.2:1b"], check=True)
print("✅ Модель готова! Теперь можно запускать RAG-код.")

Запускаем сервер Ollama...
Ждем запуска сервера...
✅ Сервер Ollama запущен!
Скачиваем модель llama3.2:1b (около 1.3 ГБ)...
✅ Модель готова! Теперь можно запускать RAG-код.


In [12]:
import subprocess
print("Скачиваем модель для эмбеддингов (nomic-embed-text)...")
subprocess.run(["ollama", "pull", "nomic-embed-text"], check=True)
print("✅ Модель для эмбеддингов готова!")

Скачиваем модель для эмбеддингов (nomic-embed-text)...
✅ Модель для эмбеддингов готова!


In [13]:
from langchain_community.llms import Ollama
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.chains import ConversationalRetrievalChain

# 1. Наш документ (база знаний)
document = """
Компания ТИСС занимается разработкой и внедрением AI-сервисов.
Основное направление работы — это развертывание локальных LLM (больших языковых моделей) на серверах компании.
Для работы используются инструменты: Ollama, Open WebUI, GPUStack, n8n и Docker.
Инженер по локальным LLM должен уверенно работать в операционной системе Linux и писать скрипты на языке Python.
Важной частью работы является создание RAG-систем (Retrieval-Augmented Generation) для чат-ботов технической поддержки.
RAG-система позволяет отвечать на вопросы пользователей, опираясь на внутреннюю базу знаний компании, а не на общие данные из интернета.
Для транскрибации (расшифровки) звонков используется модель Whisper.
Кандидат на эту должность должен иметь опыт работы с API (REST, GraphQL) и уметь обрабатывать данные в форматах JSON и XML.
График работы — 8/5, формат работы удаленный.
Обязательное условие: кандидат не должен хотеть заниматься только обучением моделей, главное — это внедрение и автоматизация готовых решений.
"""

# 2. Разбиваем на чанки
text_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
texts = text_splitter.split_text(document)

# 3. Создаем эмбеддинги (ИСПОЛЬЗУЕМ СПЕЦИАЛЬНУЮ МОДЕЛЬ) и векторное хранилище
print("Создаем эмбеддинги (это может занять минуту)...")
embeddings = OllamaEmbeddings(model="nomic-embed-text") # <-- ИСПРАВЛЕНИЕ ЗДЕСЬ
db = FAISS.from_texts(texts, embeddings)

# 4. Создаем цепочку RAG (для генерации ответа используем чат-модель)
print("Настраиваем RAG...")
llm = Ollama(model="llama3.2:1b", temperature=0.0) # <-- Для ответов оставляем llama
retriever = db.as_retriever()
rag_chain = ConversationalRetrievalChain.from_llm(llm=llm, retriever=retriever)

# 5. Тестируем!
chat_history = []
question = "Какой график работы и что нужно уметь инженеру?"
print(f"Вопрос: {question}\n")
response = rag_chain({"question": question, "chat_history": chat_history})

print(f"Ответ: {response['answer']}")

Создаем эмбеддинги (это может занять минуту)...
Настраиваем RAG...
Вопрос: Какой график работы и что нужно уметь инженеру?



/tmp/ipykernel_1197/3097591800.py:32: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm = Ollama(model="llama3.2:1b", temperature=0.0) # <-- Для ответов оставляем llama
/tmp/ipykernel_1197/3097591800.py:40: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  response = rag_chain({"question": question, "chat_history": chat_history})


Ответ: Я не знаю, что такое график работы.


In [14]:
!ollama pull qwen2.5:1.5b

In [15]:
from langchain_community.llms import Ollama
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.chains import ConversationalRetrievalChain

# 1. Наш документ (база знаний)
document = """
Компания ТИСС занимается разработкой и внедрением AI-сервисов.
Основное направление работы — это развертывание локальных LLM (больших языковых моделей) на серверах компании.
Для работы используются инструменты: Ollama, Open WebUI, GPUStack, n8n и Docker.
Инженер по локальным LLM должен уверенно работать в операционной системе Linux и писать скрипты на языке Python.
Важной частью работы является создание RAG-систем (Retrieval-Augmented Generation) для чат-ботов технической поддержки.
RAG-система позволяет отвечать на вопросы пользователей, опираясь на внутреннюю базу знаний компании, а не на общие данные из интернета.
Для транскрибации (расшифровки) звонков используется модель Whisper.
Кандидат на эту должность должен иметь опыт работы с API (REST, GraphQL) и уметь обрабатывать данные в форматах JSON и XML.
График работы — 8/5, формат работы удаленный.
Обязательное условие: кандидат не должен хотеть заниматься только обучением моделей, главное — это внедрение и автоматизация готовых решений.
"""

# 2. Разбиваем на чанки
text_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
texts = text_splitter.split_text(document)

# 3. Создаем эмбеддинги (ИСПОЛЬЗУЕМ СПЕЦИАЛЬНУЮ МОДЕЛЬ) и векторное хранилище
print("Создаем эмбеддинги (это может занять минуту)...")
embeddings = OllamaEmbeddings(model="nomic-embed-text") # <-- ИСПРАВЛЕНИЕ ЗДЕСЬ
db = FAISS.from_texts(texts, embeddings)

# 4. Создаем цепочку RAG (для генерации ответа используем чат-модель)
print("Настраиваем RAG...")
llm = Ollama(model="qwen2.5:1.5b", temperature=0.0)
retriever = db.as_retriever()
rag_chain = ConversationalRetrievalChain.from_llm(llm=llm, retriever=retriever)

# 5. Тестируем!
chat_history = []
question = "Какой график работы и что нужно уметь инженеру?"
print(f"Вопрос: {question}\n")
response = rag_chain({"question": question, "chat_history": chat_history})

print(f"Ответ: {response['answer']}")

Создаем эмбеддинги (это может занять минуту)...
Настраиваем RAG...
Вопрос: Какой график работы и что нужно уметь инженеру?

Ответ: Кандидат на эту должность должен иметь опыт работы с API (REST, GraphQL) и уметь обрабатывать данные в форматах JSON и XML.
